In [ ]:
# BLOQUE 1: Imports y setup básico

import os
import gc
import time
import math
import random

import numpy as np
import torch
from tqdm import tqdm
from datetime import datetime

print(" ADG436 – Regeneración de dataset analítico (safe RAM)")

MASTER_SEED = 42
np.random.seed(MASTER_SEED)
random.seed(MASTER_SEED)
torch.manual_seed(MASTER_SEED)

os.makedirs("datasets", exist_ok=True)
print(f"Seed global: {MASTER_SEED}")

In [ ]:
# BLOQUE 2: Modelo analítico del ADG436, versión corregida
#
# CAMBIOS respecto al original:
# ─────────────────────────────────────────────────────────────────────────────
# 1. RNG propio (np.random.Generator) → el ruido del modelo ya no contamina
#    la semilla global del generador de señales. Cada sample tiene su propio
#    rng aislado: rng = np.random.default_rng(seed).
#
# 2. Charge injection más realista:
#    - El glitch no es una muestra puntual sino un pulso exponencial decayente
#      (tau_ci ~ 5 ns) que afecta tanto a s1a como a s1b con polaridades
#      opuestas (comportamiento real del ADG436: cargas de canal en complemeto).
#    - Amplitud proporcional a la carga real: V_glitch ≈ Q_ci / C_load.
#      Con Q_ci=10 pC, C_load≈200 pF → ~50 mV (coherente con datasheet).
#
# 3. Ruido térmico (Johnson-Nyquist) calculado correctamente:
#    V_n_rms = sqrt(4 * k * T * Ron * BW)
#    Con k=1.38e-23, T=300K, Ron=4Ω, BW=5 MHz → ~0.32 µV rms
#    (el original usaba 1 µV hardcodeado sin base física).
#
# 4. Delay de switching modelado con RC de primer orden:
#    tau = ton/2.2  (tiempo de subida 10%→90% ≈ 2.2*tau para exp)
#    La señal de control pasa por este filtro antes del gating.

from scipy import signal as sp_signal

class ADG436Analytical:
    """
    Modelo analítico físicamente calibrado del ADG436 (dual SPDT).

    Parámetros nominales (datasheet Analog Devices ADG436):
      Ron     = 4 Ω          (typ @ Vdd=15V)
      ton     = 160 ns       (typ)
      toff    = 100 ns       (typ)
      Q_ci    = 10 pC        (charge injection typ)
      C_load  = 200 pF       (carga típica de laboratorio)
      T       = 300 K        (temperatura ambiente)
      BW      = fs/2         (ancho de banda del canal muestreado)
    """

    def __init__(self, fs=10e6, rload=1000.0,
                 v_threshold=2.5,
                 ron=4.0,
                 ton=160e-9,
                 toff=100e-9,
                 charge_inj=10e-12,
                 c_load=200e-12,
                 temp_k=300.0,
                 add_noise=True,
                 add_charge_injection=True):
        self.fs = fs
        self.rload = rload
        self.v_threshold = v_threshold
        self.ron = ron
        self.ton = ton
        self.toff = toff
        self.charge_inj = charge_inj
        self.c_load = c_load
        self.temp_k = temp_k
        self.add_noise = add_noise
        self.add_charge_injection = add_charge_injection

        # Ruido térmico rms: sqrt(4 * k * T * R * BW)
        k_boltzmann = 1.38e-23
        bw = fs / 2.0
        self._noise_std = np.sqrt(4 * k_boltzmann * temp_k * ron * bw)

        # Amplitud pico del glitch de charge injection (V)
        self._v_glitch = charge_inj / c_load   # ~50 mV

        # Tau del filtro RC de switching (ton/toff → tau_on/tau_off)
        # Para un RC: t_rise(10%→90%) ≈ 2.197 * tau
        self._tau_on  = ton  / 2.197
        self._tau_off = toff / 2.197

        self._build_switch_filter()

    def _build_switch_filter(self):
        """
        Construye dos filtros RC: uno para el flanco de subida (ton)
        y otro para el de bajada (toff). El switch real tiene tiempos
        asimétricos (ton ≠ toff en el ADG436).
        """
        def make_rc_kernel(tau, fs, n_tau=8):
            length = max(int(n_tau * tau * fs), 4)
            t = np.arange(length) / fs
            h = np.exp(-t / tau)
            h /= h.sum()
            return h

        self._h_on  = make_rc_kernel(self._tau_on,  self.fs)
        self._h_off = make_rc_kernel(self._tau_off, self.fs)

    def _apply_asymmetric_switching(self, switch_binary):
        """
        Aplica el filtro RC de forma asimétrica según el flanco:
        flancos de subida → tau_on, flancos de bajada → tau_off.
        Esto reproduce el comportamiento físico donde ton ≠ toff.
        """
        n = len(switch_binary)
        result = np.zeros(n)
        state = float(switch_binary[0])

        for i in range(n):
            target = float(switch_binary[i])
            # Usamos el alpha correspondiente al flanco actual
            if target > state:
                alpha = 1.0 - np.exp(-1.0 / (self._tau_on * self.fs))
            else:
                alpha = 1.0 - np.exp(-1.0 / (self._tau_off * self.fs))
            state += alpha * (target - state)
            result[i] = state

        return np.clip(result, 0.0, 1.0)

    def simulate(self, v_analog, v_ctrl, rng=None):
        """
        v_analog, v_ctrl : np.array 1D de la misma longitud.
        rng              : np.random.Generator aislado (NO usa estado global).
        Devuelve:
          s1a, s1b: salidas del primer SPDT (A/B) como float32.
        """
        if rng is None:
            rng = np.random.default_rng()

        n = len(v_analog)

        # 1) Lógica digital con histéresis leve (±50 mV) para evitar
        #    chatter en señales de control ruidosas (comportamiento real).
        v_hi = self.v_threshold + 0.05
        v_lo = self.v_threshold - 0.05
        switch = np.zeros(n)
        state = int(v_ctrl[0] > self.v_threshold)
        for i in range(n):
            if state == 0 and v_ctrl[i] > v_hi:
                state = 1
            elif state == 1 and v_ctrl[i] < v_lo:
                state = 0
            switch[i] = state

        # 2) Atenuación por Ron (divisor resistivo con Rload)
        att = self.rload / (self.ron + self.rload)
        v_att = v_analog * att

        # 3) Switching asimétrico (ton ≠ toff)
        sw_delayed = self._apply_asymmetric_switching(switch)

        # 4) Gating
        s1a = v_att * sw_delayed
        s1b = v_att * (1.0 - sw_delayed)

        # 5) Charge injection – pulso exponencial decayente en cada flanco
        if self.add_charge_injection:
            edges = np.diff(switch, prepend=switch[0])
            idx_edges = np.where(edges != 0)[0]

            # Tau del glitch de charge injection (~5 ns, típico ADG436)
            tau_ci = 5e-9
            len_ci = max(int(10 * tau_ci * self.fs), 3)
            t_ci = np.arange(len_ci) / self.fs
            h_ci = np.exp(-t_ci / tau_ci)

            for idx in idx_edges:
                sign = np.sign(edges[idx])
                # Variación del ±20% en amplitud (dispersión de fabricación)
                amp = self._v_glitch * sign * rng.uniform(0.8, 1.2)
                end_ci = min(idx + len_ci, n)
                length = end_ci - idx
                s1a[idx:end_ci] += amp * h_ci[:length]
                s1b[idx:end_ci] -= amp * h_ci[:length]   # polaridad opuesta

        # 6) Ruido térmico – RNG aislado, std calculado físicamente
        if self.add_noise:
            s1a += rng.standard_normal(n) * self._noise_std
            s1b += rng.standard_normal(n) * self._noise_std

        # 7) Saturación física (rails del ADG436: ±Vss)
        s1a = np.clip(s1a, -5.0, 5.0)
        s1b = np.clip(s1b, -5.0, 5.0)

        return s1a.astype(np.float32), s1b.astype(np.float32)


print(" ADG436Analytical definido")
print(f"   Ruido térmico rms: {np.sqrt(4*1.38e-23*300*4*5e6)*1e6:.3f} µV")
print(f"   Glitch CI pico:    {10e-12/200e-12*1e3:.1f} mV")
print(f"   tau_on:            {160e-9/2.197*1e9:.1f} ns")
print(f"   tau_off:           {100e-9/2.197*1e9:.1f} ns")


In [ ]:
# BLOQUE 3: Generador de señales para ADG436, versión corregida
#
# CAMBIOS:
# 1. Usa np.random.Generator (rng) propio en vez de np.random.seed global.
#    Cada sample recibe su rng derivado del seed maestro con SeedSequence,
#    garantizando independencia estadística y reproducibilidad perfecta.
#
# 2. Señales de control digitales más realistas:
#    - PRBS con frecuencias ajustadas al rango típico de uso (1 Hz – 100 kHz).
#    - Pulse train con tiempos mínimos de estabilidad ≥ 5*ton (800 ns) para
#      que el switch tenga tiempo de completar la transición entre pulsos.
#      (Si el pulso es más corto que ton el switch nunca llega a conducir
#      completamente → comportamiento no capturado en el original.)
#
# 3. Señales analógicas dentro del rango de entrada válido del ADG436:
#    Vin_max = ±Vdd - 2V ≈ ±3V con supply ±5V.
#    Las amplitudes se limitan a 0.9 * 3V = 2.7 V pico.
#
# 4. Se agrega señal analógica DC+AC para cubrir el caso de offset de DC
#    (común en mediciones reales con el ADG436 como multiplexor).

class ADG436SignalGen:
    # Rango de entrada válido del ADG436 con ±5V supply
    V_IN_MAX = 2.7   # V pico máximo (90% del rail)
    # Tiempo mínimo estable del control para que el switch conmute completamente
    T_STABLE_MIN = 5 * 160e-9   # 5 * ton = 800 ns

    def __init__(self, fs, duration):
        self.fs = fs
        self.duration = duration
        self.n = int(fs * duration)
        self.t = np.linspace(0.0, duration, self.n, endpoint=False)
        # NO almacena semilla: el rng se pasa en cada llamada

    # Señales analógicas

    def multitone(self, rng):
        n_tones = rng.integers(3, 9)
        freqs  = rng.uniform(10e3, min(500e3, self.fs * 0.45), n_tones)
        amps   = rng.uniform(0.05, 0.4, n_tones)
        phases = rng.uniform(0, 2*np.pi, n_tones)
        y = sum(a * np.sin(2*np.pi*f*self.t + p)
                for f, a, p in zip(freqs, amps, phases))
        peak = np.max(np.abs(y)) + 1e-12
        y = y / peak * self.V_IN_MAX * rng.uniform(0.3, 1.0)
        return y.astype(np.float32), "Multitone"

    def chirp(self, rng):
        f0 = rng.uniform(1e3, 50e3)
        f1 = rng.uniform(200e3, min(1e6, self.fs * 0.45))
        amp = rng.uniform(0.2, 0.9) * self.V_IN_MAX
        y = sp_signal.chirp(self.t, f0=f0, f1=f1, t1=self.duration,
                            method="logarithmic")
        return (amp * y).astype(np.float32), "Chirp Log"

    def pink_noise(self, rng):
        white = rng.standard_normal(self.n)
        fft = np.fft.rfft(white)
        freqs = np.fft.rfftfreq(self.n, 1/self.fs)
        freqs[0] = 1.0
        fft /= np.sqrt(freqs)
        y = np.fft.irfft(fft, n=self.n)
        y /= (np.std(y) + 1e-12)
        amp = rng.uniform(0.05, 0.2) * self.V_IN_MAX
        y *= amp
        return y.astype(np.float32), "Pink Noise"

    def sine(self, rng):
        f   = rng.uniform(100e3, min(1e6, self.fs * 0.45))
        amp = rng.uniform(0.2, 1.0) * self.V_IN_MAX
        phi = rng.uniform(0, 2*np.pi)
        y = amp * np.sin(2*np.pi*f*self.t + phi)
        return y.astype(np.float32), f"Sine {int(f/1e3)}kHz"

    def dc_plus_ac(self, rng):
        """DC con ripple – caso típico en multiplexado de sensores."""
        dc  = rng.uniform(-2.0, 2.0)
        f   = rng.uniform(1e3, 100e3)
        amp = rng.uniform(0.05, 0.3) * self.V_IN_MAX
        y   = dc + amp * np.sin(2*np.pi*f*self.t)
        y   = np.clip(y, -self.V_IN_MAX, self.V_IN_MAX)
        return y.astype(np.float32), "DC+AC"

    def impulse(self, rng):
        y = np.zeros(self.n, dtype=np.float32)
        n_imp = rng.integers(1, 6)
        idx  = rng.integers(0, self.n, n_imp)
        amps = rng.uniform(0.3, 1.0, n_imp) * self.V_IN_MAX
        y[idx] = amps
        return y, "Impulse"

    # Señales de control digitales

    def prbs(self, rng, switch_freq=None):
        """
        PRBS con constraint de tiempo mínimo estable:
        cada bit dura al menos T_STABLE_MIN para que el switch
        tenga tiempo de completar su transición.
        """
        min_samples_per_bit = max(int(self.T_STABLE_MIN * self.fs), 1)

        if switch_freq is None:
            # Rango: 100 Hz – fs/(2*min_samples_per_bit)
            f_max = self.fs / (2 * min_samples_per_bit)
            f_max = max(f_max, 101.0)
            switch_freq = rng.uniform(100, min(10_000, f_max))

        period = 1.0 / switch_freq
        samples_per_bit = max(int(self.fs * period), min_samples_per_bit)
        n_bits = max(int(self.n / samples_per_bit) + 1, 3)

        bits = rng.integers(0, 2, n_bits)
        sig  = np.repeat(bits, samples_per_bit)[:self.n]
        if len(sig) < self.n:
            sig = np.pad(sig, (0, self.n - len(sig)), mode="edge")

        return (sig * 5.0).astype(np.float32), f"PRBS {n_bits}b {int(switch_freq)}Hz"

    def prbs_fast(self, rng):
        min_samp = max(int(self.T_STABLE_MIN * self.fs), 1)
        f_max = self.fs / (2 * min_samp)
        f_max = max(f_max, 1001.0)
        return self.prbs(rng, switch_freq=rng.uniform(1_000, min(100_000, f_max)))

    def pulse_train(self, rng):
        """
        Tren de pulsos con período mínimo respetado.
        Duty cycle entre 20%-80%.
        """
        min_samp = max(int(self.T_STABLE_MIN * self.fs), 1)
        f_max = self.fs / (2 * min_samp)
        f = rng.uniform(100, min(5_000, f_max))
        duty = rng.uniform(0.2, 0.8)
        y = sp_signal.square(2*np.pi*f*self.t, duty=duty)
        y = (y + 1.0) * 2.5   # mapear [-1,1] → [0,5] V
        return y.astype(np.float32), "Pulse Train"

    # Selección aleatoria

    def get_analog(self, rng):
        choices = ["multitone", "chirp", "pink_noise", "sine", "dc_plus_ac", "impulse"]
        choice  = choices[rng.integers(0, len(choices))]
        return getattr(self, choice)(rng)

    def get_digital(self, rng):
        choices = ["prbs", "prbs_fast", "pulse_train"]
        choice  = choices[rng.integers(0, len(choices))]
        return getattr(self, choice)(rng)


print(" ADG436SignalGen definido")
print(f"   V_IN_MAX     : {ADG436SignalGen.V_IN_MAX} V")
print(f"   T_STABLE_MIN : {ADG436SignalGen.T_STABLE_MIN*1e6:.2f} µs  (5 × ton)")


In [ ]:
# @title
# BLOQUE 4: Generador de dataset por batches – versión corregida
#
# CAMBIOS:
# 1. SeedSequence de NumPy: cada sample_id produce un rng independiente
#    y reproducible sin contaminar el estado global. Garantiza que ejecutar
#    la generación en cualquier orden produce el mismo resultado.
#
# 2. El rng se pasa explícitamente al modelo (.simulate) y al generador de
#    señales (.get_analog / .get_digital), eliminando cualquier dependencia
#    de np.random.seed global.
#
# 3. Las stats de normalización se calculan sobre el dataset COMPLETO
#    (recorriendo todos los batches) y se guardan en el metadata → las
#    stats son deterministas, únicas y guardadas con el modelo.

class ADG436BatchedGenerator:
    def __init__(self, fs=10e6, duration=0.001, seed=42):
        self.fs = fs
        self.duration = duration
        self.seed = seed
        self.model = ADG436Analytical(fs=fs, rload=1000.0)
        self.sig_gen = ADG436SignalGen(fs=fs, duration=duration)
        # SeedSequence maestra – produce child seeds independientes
        self._ss = np.random.SeedSequence(seed)

    def generate_sample(self, sample_id):
        """
        Genera un sample con un rng completamente aislado derivado
        de la SeedSequence maestra + sample_id.
        Reproducible e independiente del orden de generación.
        """
        child_seed = self._ss.spawn(1)[0]
        # Incorporar sample_id en el estado para que cada sample sea único
        rng = np.random.default_rng([child_seed.entropy, sample_id])

        v_analog, type_a = self.sig_gen.get_analog(rng)
        v_ctrl,   type_d = self.sig_gen.get_digital(rng)

        # El modelo recibe su propio rng → ruido térmico aislado
        s1a, s1b = self.model.simulate(v_analog, v_ctrl, rng=rng)

        return {
            "id": sample_id,
            "x": {
                "v_analog":      v_analog,
                "v_ctrl":        v_ctrl,
                "v_analog_type": type_a,
                "v_ctrl_type":   type_d,
            },
            "y": {
                "s1a": s1a,
                "s1b": s1b,
            },
            "metadata": {
                "fs":       self.fs,
                "duration": self.duration,
                "n_samples": len(v_analog),
            }
        }

    def _compute_full_stats(self, batch_files):
        """
        Calcula mean/std sobre TODOS los samples del dataset.
        Determinista: no usa np.random en ningún punto.
        """
        print(" Calculando stats de normalización (dataset completo)...")
        sum_  = {"v_analog": 0.0, "v_ctrl": 0.0, "s1a": 0.0, "s1b": 0.0}
        sum2_ = {"v_analog": 0.0, "v_ctrl": 0.0, "s1a": 0.0, "s1b": 0.0}
        count = 0

        for bf in tqdm(batch_files, desc="Stats pass"):
            bd = torch.load(bf, weights_only=False)
            for s in bd["samples"]:
                for key, arr in [("v_analog", s["x"]["v_analog"]),
                                  ("v_ctrl",   s["x"]["v_ctrl"]),
                                  ("s1a",      s["y"]["s1a"]),
                                  ("s1b",      s["y"]["s1b"])]:
                    sum_[key]  += float(np.sum(arr))
                    sum2_[key] += float(np.sum(arr**2))
                count += len(s["x"]["v_analog"])

        stats = {}
        for key in sum_:
            mean = sum_[key] / count
            std  = np.sqrt(sum2_[key]/count - mean**2)
            stats[key] = {"mean": float(mean), "std": float(std) + 1e-8}

        print(" Stats calculadas sobre", count, "puntos")
        for k, v in stats.items():
            print(f"   {k:10s}  mean={v['mean']:+.4f}  std={v['std']:.4f}")
        return stats

    def generate_batched(self,
                         n_samples=20_000,
                         batch_size=1000,
                         out_dir="datasets",
                         metadata_name="adg436_20k_metadata_v2.pt"):
        os.makedirs(out_dir, exist_ok=True)
        batch_dir = os.path.join(out_dir, "batches")
        os.makedirs(batch_dir, exist_ok=True)

        n_batches = math.ceil(n_samples / batch_size)
        print("="*70)
        print("GENERACIÓN ADG436 ANALÍTICO POR BATCHES")
        print("="*70)
        print(f"Total samples : {n_samples:,}")
        print(f"Batch size    : {batch_size}")
        print(f"N batches     : {n_batches}")
        print(f"Fs            : {self.fs/1e6:.1f} MHz")
        print(f"Duración      : {self.duration*1e3:.1f} ms (~{int(self.fs*self.duration):,} pts)")
        print("="*70)

        batch_files = []
        global_start = time.time()

        for b in range(n_batches):
            start_id = b * batch_size
            end_id   = min((b+1) * batch_size, n_samples)
            batch_n  = end_id - start_id

            samples_batch = [self.generate_sample(sid)
                             for sid in range(start_id, end_id)]

            batch_path = os.path.join(batch_dir, f"batch_{b:04d}.pt")
            torch.save({"samples":  samples_batch,
                        "batch_id": b,
                        "start_id": start_id,
                        "end_id":   end_id}, batch_path)
            batch_files.append(batch_path)

            del samples_batch
            gc.collect()

            elapsed = time.time() - global_start
            print(f"Batch {b+1}/{n_batches} guardado ({batch_n} samples) "
                  f"| t acumulado: {elapsed:.1f}s")

        # Stats deterministas sobre el dataset completo
        stats = self._compute_full_stats(batch_files)

        metadata = {
            "n_samples":   n_samples,
            "n_batches":   n_batches,
            "batch_size":  batch_size,
            "batch_files": batch_files,
            "fs":          self.fs,
            "duration":    self.duration,
            "seed":        self.seed,
            "stats":       stats,          # ← guardadas en metadata
            "created_at":  datetime.now().isoformat()
        }
        meta_path = os.path.join(out_dir, metadata_name)
        torch.save(metadata, meta_path)

        total_elapsed = time.time() - global_start
        print("="*70)
        print("GENERACIÓN COMPLETADA")
        print("="*70)
        print(f"Tiempo total: {total_elapsed:.1f}s (~{total_elapsed/60:.1f} min)")
        print(f"Metadata    : {meta_path}")
        print("="*70)

        return meta_path


print(" ADG436BatchedGenerator definido")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# @title
# BLOQUE 5: Lanzar regeneración del dataset

generator = ADG436BatchedGenerator(fs=10e6, duration=0.001, seed=42)

META_PATH = generator.generate_batched(
    n_samples=20_000,
    batch_size=1000,
    out_dir="/content/drive/MyDrive/universal_filter_ngspice/datasets",
    metadata_name="adg436_20k_metadata_v2.pt"
)

print(f"\nMetadata final en: {META_PATH}")


In [ ]:
# BLOQUE 6: ADG436DatasetRAM, dataset en RAM compatible con el resto del notebook
#
# Todos los atributos del dataset lazy original se preservan para que
# las celdas de verificación, visualización y LTspice funcionen sin cambios:
#   .split_indices  → array de ids de samples del split
#   .window_index   → lista de (batch_idx, local_idx, start) por ventana
#   .metadata       → dict con fs, stats, batch_files, etc.
#   .window_size    → tamaño de ventana
#   ._get_batch()   → carga el batch desde disco (solo para visualización)

import os
import math
import torch
import numpy as np
from torch.utils.data import Dataset
from tqdm import tqdm


class ADG436DatasetRAM(Dataset):
    """
    Carga TODAS las ventanas en RAM al inicio.
    Mantiene los atributos del dataset lazy para compatibilidad
    con las celdas de verificación y visualización.
    """

    def __init__(self, windows, split_indices, window_index, metadata, window_size):
        self.windows       = windows        # lista de (x_tensor, y_tensor)
        self.split_indices = split_indices  # np.array de sample ids en este split
        self.window_index  = window_index   # lista de (batch_idx, local_idx, start)
        self.metadata      = metadata       # dict original del .pt
        self.window_size   = window_size
        self._batch_cache  = {}             # caché ligero para _get_batch

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]

    def _get_batch(self, batch_idx):
        """Carga un batch del disco (igual que el dataset lazy). Solo para visualización."""
        if batch_idx not in self._batch_cache:
            bf = self.metadata["batch_files"][batch_idx]
            bd = torch.load(bf, weights_only=False)
            # Guardar solo el último batch para no saturar RAM
            self._batch_cache = {batch_idx: bd["samples"]}
        return self._batch_cache[batch_idx]


def build_dataset_ram(metadata_file, window_size=200, stride=500,
                      split=None, train_frac=0.70, val_frac=0.15, seed=42):
    metadata    = torch.load(metadata_file, weights_only=False)
    stats       = metadata["stats"]
    batch_files = metadata["batch_files"]
    n_total     = metadata["n_samples"]
    batch_size  = metadata["batch_size"]

    # Split por sample
    rng = np.random.default_rng(seed)
    idx = np.arange(n_total)
    rng.shuffle(idx)
    n_train = int(train_frac * n_total)
    n_val   = int(val_frac   * n_total)
    splits  = {
        "train": (idx[:n_train],                  set(idx[:n_train].tolist())),
        "val":   (idx[n_train:n_train+n_val],     set(idx[n_train:n_train+n_val].tolist())),
        "test":  (idx[n_train+n_val:],            set(idx[n_train+n_val:].tolist())),
        None:    (idx,                             set(idx.tolist())),
    }
    split_indices_arr, allowed = splits[split]

    windows      = []
    window_index = []   # (batch_idx, local_idx_en_batch, start)

    print(f"Cargando split=\'{split}\' en RAM...")

    for bf_idx, bf in enumerate(tqdm(batch_files, desc="Batches")):
        bd = torch.load(bf, weights_only=False)
        for local_idx, s in enumerate(bd["samples"]):
            if s["id"] not in allowed:
                continue

            n  = s["metadata"]["n_samples"]
            va = s["x"]["v_analog"]
            vc = s["x"]["v_ctrl"]
            y1 = s["y"]["s1a"]
            y2 = s["y"]["s1b"]

            # Normalizar
            va = (va - stats["v_analog"]["mean"]) / stats["v_analog"]["std"]
            vc = (vc - stats["v_ctrl"]["mean"])   / stats["v_ctrl"]["std"]
            y1 = (y1 - stats["s1a"]["mean"])      / stats["s1a"]["std"]
            y2 = (y2 - stats["s1b"]["mean"])      / stats["s1b"]["std"]

            # Ventanas
            for start in range(0, n - window_size + 1, stride):
                end = start + window_size
                x_w = np.stack([va[start:end], vc[start:end]], axis=1).astype(np.float32)
                y_w = np.stack([y1[start:end], y2[start:end]], axis=1).astype(np.float32)
                windows.append((torch.from_numpy(x_w), torch.from_numpy(y_w)))
                window_index.append((bf_idx, local_idx, start))

    print(f"  → {len(windows):,} ventanas cargadas")

    return ADG436DatasetRAM(
        windows       = windows,
        split_indices = split_indices_arr,
        window_index  = window_index,
        metadata      = metadata,
        window_size   = window_size,
    )


META_PATH = "/content/drive/MyDrive/universal_filter_ngspice/datasets/adg436_20k_metadata_v2.pt"

train_dataset = build_dataset_ram(META_PATH, window_size=200, stride=500, split="train")
val_dataset   = build_dataset_ram(META_PATH, window_size=200, stride=500, split="val")
test_dataset  = build_dataset_ram(META_PATH, window_size=200, stride=500, split="test")


In [ ]:
# BLOQUE 7: Verificación de calidad del dataset

print("VERIFICACIÓN DE CALIDAD DEL DATASET")

# 1. Sin leakage entre splits
train_set = set(train_dataset.split_indices.tolist())
val_set   = set(val_dataset.split_indices.tolist())
test_set  = set(test_dataset.split_indices.tolist())

print("\n  Leakage entre splits (deben ser 0):")
print(f"   Train ∩ Val  : {len(train_set & val_set)} samples")
print(f"   Train ∩ Test : {len(train_set & test_set)} samples")
print(f"   Val ∩ Test   : {len(val_set & test_set)} samples")
assert not (train_set & val_set) and not (train_set & test_set) and not (val_set & test_set)
print("   Sin leakage")

# 2. Diversidad de ventanas en test (usando datos ya cargados en RAM)
print("\n Diversidad entre ventanas del test:")

rng_check = np.random.default_rng(0)
n_check   = min(8, len(test_dataset))
chosen    = rng_check.choice(len(test_dataset), n_check, replace=False)

def fingerprint_window(x_t, y_t):
    """Fingerprint sobre tensores de ventana (ya normalizados)."""
    arrs = [x_t[:, 0].numpy(), x_t[:, 1].numpy(),
            y_t[:, 0].numpy(), y_t[:, 1].numpy()]
    feats = []
    for a in arrs:
        feats.extend([a.mean(), a.std(), a.max(), a.min(),
                      np.sqrt(np.mean(a**2)), np.percentile(np.abs(a), 95)])
    return np.array(feats)

fps = []
for i in chosen:
    x_t, y_t = test_dataset[int(i)]
    fps.append(fingerprint_window(x_t, y_t))

diffs = [np.linalg.norm(fps[i] - fps[j])
         for i in range(len(fps)) for j in range(i+1, len(fps))]

print(f"   Diferencia media entre fingerprints: {np.mean(diffs):.4f}")
print(f"   Diferencia mínima:                   {np.min(diffs):.4f}")
if np.min(diffs) < 1e-3:
    print("  Algunos samples son casi idénticos – revisar generación")
else:
    print(" Samples suficientemente diversos")

# 3. Tamaños de splits
print("\n3  Tamaño de cada split (ventanas):")
print(f"   Train : {len(train_dataset):,} ventanas")
print(f"   Val   : {len(val_dataset):,} ventanas")
print(f"   Test  : {len(test_dataset):,} ventanas")
print(f"   Total : {len(train_dataset)+len(val_dataset)+len(test_dataset):,} ventanas")
print("\n Verificación completada")


In [ ]:
# BLOQUE 8: Visualización rápida de samples, comprobación visual física

import matplotlib.pyplot as plt

metadata = torch.load("/content/drive/MyDrive/universal_filter_ngspice/datasets/adg436_20k_metadata_v2.pt", weights_only=False)
fs    = metadata["fs"]
stats = metadata["stats"]

print("Stats de normalización (del metadata, calculadas sobre dataset completo):")
for k, v in stats.items():
    print(f"  {k:10s}  mean={v['mean']:+.5f}  std={v['std']:.5f}")

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.suptitle("ADG436 – Muestras en escala física (V)", fontsize=13)

rng_vis = np.random.default_rng(7)
chosen_vis = rng_vis.choice(train_dataset.split_indices, 3, replace=False)

for col, g_idx in enumerate(chosen_vis):
    s = load_sample_lazy(train_dataset, g_idx)

    n    = s["metadata"]["n_samples"]
    t    = np.arange(n) / fs * 1e6
    v_a  = s["x"]["v_analog"]
    v_c  = s["x"]["v_ctrl"]
    s1a  = s["y"]["s1a"]
    s1b  = s["y"]["s1b"]
    label_a = s["x"]["v_analog_type"]
    label_d = s["x"]["v_ctrl_type"]

    axes[0, col].plot(t, v_a, lw=0.8, color="C0")
    axes[0, col].set_title(f"Sample {g_idx}\nv_analog: {label_a}", fontsize=9)
    axes[0, col].set_ylabel("V_analog [V]")
    axes[0, col].set_ylim(-3.2, 3.2)
    axes[0, col].axhline(0, color="k", lw=0.3, ls="--")
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(t, v_c, lw=0.8, color="C1")
    axes[1, col].set_title(f"v_ctrl: {label_d}", fontsize=9)
    axes[1, col].set_ylabel("V_ctrl [V]")
    axes[1, col].set_ylim(-0.5, 5.5)
    axes[1, col].axhline(2.5, color="r", lw=0.5, ls="--", label="Vth=2.5V")
    axes[1, col].legend(fontsize=7)
    axes[1, col].grid(True, alpha=0.3)

    axes[2, col].plot(t, s1a, lw=0.8, color="C2", label="S1A")
    axes[2, col].plot(t, s1b, lw=0.8, color="C3", label="S1B", alpha=0.8)
    axes[2, col].set_ylabel("Salida [V]")
    axes[2, col].set_xlabel("Tiempo [µs]")
    axes[2, col].set_ylim(-3.5, 3.5)
    axes[2, col].legend(fontsize=7)
    axes[2, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(" Los rangos deben ser coherentes con ADG436: |S1A|,|S1B| ≤ ~2.7 V")


# Entrenamiento LSTM

In [ ]:
# BLOQUE 9: Arquitectura LSTM + configuración de entrenamiento

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

class ADG436LSTM(nn.Module):
    """
    LSTM para emulación del ADG436.
    Entrada : [v_analog, v_ctrl]  → 2 features
    Salida  : [s1a, s1b]          → 2 outputs
    """
    def __init__(self, input_size=2, hidden_size=128,
                 num_layers=2, dropout=0.2, output_size=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out)

# Hiperparámetros
BATCH_SIZE    = 128
NUM_EPOCHS    = 60
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")

# num_workers=0: necesario con lazy loading + caché compartido en proceso principal
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(device.type == "cuda"),
                          drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

model = ADG436LSTM().to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParámetros LSTM: {num_params:,}")

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                   factor=0.5, patience=5)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total = 0.0
    for x, y in tqdm(loader, desc="Train", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Val", leave=False):
            x, y = x.to(device), y.to(device)
            total += criterion(model(x), y).item()
    return total / len(loader)

In [ ]:
# BLOQUE 10: Loop de entrenamiento

# Stats del metadata (deterministas, no del dataset object)
metadata  = torch.load("/content/drive/MyDrive/universal_filter_ngspice/datasets/adg436_20k_metadata_v2.pt", weights_only=False)
stats_meta = metadata["stats"]

train_losses, val_losses = [], []
best_val_loss = float("inf")
best_epoch    = 0
start_time    = time.time()

print("=" * 70)
print("ENTRENAMIENTO ADG436 LSTM")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    tl = train_epoch(model, train_loader, criterion, optimizer, device)
    vl = validate_epoch(model, val_loader, criterion, device)
    train_losses.append(tl)
    val_losses.append(vl)
    scheduler.step(vl)
    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch:3d}/{NUM_EPOCHS}] Train: {tl:.8f} | Val: {vl:.8f} | LR: {lr:.6f}")

    if vl < best_val_loss:
        best_val_loss = vl
        best_epoch    = epoch
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss":           tl,
            "val_loss":             vl,
            "stats":                stats_meta,
            "input_size":           2,
            "output_size":          2,
            "num_params":           num_params,
        }, "/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_lstm_best.pt")
        print(f"    Mejor modelo guardado (Val: {vl:.8f})")

elapsed = time.time() - start_time
print(f"\n Entrenamiento completado en {elapsed/60:.2f} min")
print(f"   Mejor época   : {best_epoch}/{NUM_EPOCHS}")
print(f"   Mejor Val Loss: {best_val_loss:.8f}")

# Curva de aprendizaje
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(train_losses, label="Train Loss")
ax.semilogy(val_losses,   label="Val Loss")
ax.axvline(best_epoch - 1, color="r", ls="--", label=f"Best epoch {best_epoch}")
ax.set_xlabel("Época")
ax.set_ylabel("MSE (log)")
ax.set_title("Curva de aprendizaje ADG436 LSTM")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# BLOQUE 11: Evaluación en test set
#
# Usa test_loader (del split "test", sin overlap con train/val).
# Las stats de normalización se cargan del checkpoint para garantizar
# que la desnormalización usa exactamente los mismos valores del entrenamiento.

ckpt = torch.load("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_lstm_best.pt", weights_only=False, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

stats = ckpt["stats"]
print(f"Checkpoint época  : {ckpt['epoch']}")
print(f"Val loss (norm)   : {ckpt['val_loss']:.8f}")

mse_sum  = 0.0
n_pts    = 0
mse_on_a = mse_off_a = mse_on_b = mse_off_b = 0.0
n_on     = n_off = 0
y_true_list, y_pred_list = [], []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Test"):
        x, y  = x.to(device), y.to(device)
        yp    = model(x)

        mse_sum += nn.functional.mse_loss(yp, y, reduction="sum").item()
        n_pts   += y.numel()

        ctrl = x[:, :, 1]
        mon  = ctrl >  0.0
        moff = ctrl <= 0.0

        da = (yp[:,:,0] - y[:,:,0])**2
        db = (yp[:,:,1] - y[:,:,1])**2

        mse_on_a  += da[mon].sum().item()
        mse_off_a += da[moff].sum().item()
        mse_on_b  += db[mon].sum().item()
        mse_off_b += db[moff].sum().item()
        n_on  += mon.sum().item()
        n_off += moff.sum().item()

        y_true_list.append(y.cpu().numpy())
        y_pred_list.append(yp.cpu().numpy())

mse_norm  = mse_sum / n_pts
mse_on_a  /= max(n_on,  1)
mse_off_a /= max(n_off, 1)
mse_on_b  /= max(n_on,  1)
mse_off_b /= max(n_off, 1)

# MSE en escala física (desnormalizando)
std_s1a    = stats["s1a"]["std"]
std_s1b    = stats["s1b"]["std"]
mse_phys_a = mse_norm * std_s1a**2
mse_phys_b = mse_norm * std_s1b**2

# R²
y_true_all = np.concatenate(y_true_list).reshape(-1, 2)
y_pred_all = np.concatenate(y_pred_list).reshape(-1, 2)

def r2(y, yh):
    ss_res = np.sum((y - yh)**2)
    ss_tot = np.sum((y - y.mean())**2) + 1e-12
    return 1.0 - ss_res / ss_tot

r2_a = r2(y_true_all[:,0], y_pred_all[:,0])
r2_b = r2(y_true_all[:,1], y_pred_all[:,1])

print("\n" + "="*70)
print("RESULTADOS EN TEST SET")
print("="*70)
print(f"MSE global (norm)       : {mse_norm:.6e}")
print(f"MSE S1A (física, V²)    : {mse_phys_a:.6e}  → RMSE = {np.sqrt(mse_phys_a)*1e3:.3f} mV")
print(f"MSE S1B (física, V²)    : {mse_phys_b:.6e}  → RMSE = {np.sqrt(mse_phys_b)*1e3:.3f} mV")
print(f"R² S1A                  : {r2_a:.6f}")
print(f"R² S1B                  : {r2_b:.6f}")
print(f"\nMSE por estado (norm):")
print(f"  S1A ON  : {mse_on_a:.6e}")
print(f"  S1A OFF : {mse_off_a:.6e}")
print(f"  S1B ON  : {mse_on_b:.6e}")
print(f"  S1B OFF : {mse_off_b:.6e}")
if mse_off_a > 0:
    print(f"  Ratio ON/OFF S1A: {mse_on_a/mse_off_a:.2f}")
print("="*70)


In [ ]:
# BLOQUE 12: Visualización de predicciones en escala física – LSTM
#
# Con DatasetRAM usamos window_index + _get_batch para buscar
# ventanas CON y SIN switching, igual que antes.

def denorm(arr, key, stats):
    return arr * stats[key]["std"] + stats[key]["mean"]

def plot_prediction_physical(v_a_norm, v_c_norm, y_real_norm, y_pred_norm,
                              stats, fs, title=""):
    v_a   = denorm(v_a_norm,         "v_analog", stats)
    v_c   = denorm(v_c_norm,         "v_ctrl",   stats)
    s1a_r = denorm(y_real_norm[:,0], "s1a",      stats)
    s1b_r = denorm(y_real_norm[:,1], "s1b",      stats)
    s1a_p = denorm(y_pred_norm[:,0], "s1a",      stats)
    s1b_p = denorm(y_pred_norm[:,1], "s1b",      stats)

    T = len(v_a)
    t = np.arange(T) / fs * 1e6

    fig, axes = plt.subplots(3, 1, figsize=(13, 8))
    axes[0].plot(t, v_a, color="C0", lw=0.9, label="v_analog")
    axes[0].set_ylabel("v_analog [V]"); axes[0].set_ylim(-3.5, 3.5)
    axes[0].axhline(0, color="k", lw=0.3, ls="--")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(t, v_c, color="C1", lw=0.9, label="v_ctrl")
    axes[1].axhline(2.5, color="r", lw=0.6, ls="--", label="Vth=2.5V")
    axes[1].set_ylabel("v_ctrl [V]"); axes[1].set_ylim(-0.3, 5.3)
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(t, s1a_r, color="C2", lw=1.2, label="S1A real")
    axes[2].plot(t, s1a_p, color="C2", lw=1.0, ls="--", alpha=0.8, label="S1A pred")
    axes[2].plot(t, s1b_r, color="C3", lw=1.2, label="S1B real")
    axes[2].plot(t, s1b_p, color="C3", lw=1.0, ls="--", alpha=0.8, label="S1B pred")
    axes[2].set_ylabel("Salida [V]"); axes[2].set_xlabel("Tiempo [µs]")
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=11)
    plt.tight_layout(); plt.show()


def buscar_y_plotear(model, dataset, ckpt, device, titulo_prefix):
    """Busca ventanas CON y SIN switching en dataset y las plotea."""
    model.eval()
    fs_v    = dataset.metadata["fs"]
    stats_v = ckpt["stats"]
    found_no_sw = found_sw = False

    for win_entry in dataset.window_index:
        if found_no_sw and found_sw:
            break

        batch_idx, local_idx, start = win_entry
        end = start + dataset.window_size

        s       = dataset._get_batch(batch_idx)[local_idx]
        v_c_raw = s["x"]["v_ctrl"][start:end]
        rng_vc  = float(v_c_raw.max() - v_c_raw.min())

        va = s["x"]["v_analog"][start:end].copy()
        vc = s["x"]["v_ctrl"][start:end].copy()
        y1 = s["y"]["s1a"][start:end].copy()
        y2 = s["y"]["s1b"][start:end].copy()

        va = (va - stats_v["v_analog"]["mean"]) / stats_v["v_analog"]["std"]
        vc = (vc - stats_v["v_ctrl"]["mean"])   / stats_v["v_ctrl"]["std"]
        y1 = (y1 - stats_v["s1a"]["mean"])      / stats_v["s1a"]["std"]
        y2 = (y2 - stats_v["s1b"]["mean"])      / stats_v["s1b"]["std"]

        x_t = torch.from_numpy(np.stack([va, vc], axis=1)).float().unsqueeze(0).to(device)
        y_t = np.stack([y1, y2], axis=1)

        with torch.no_grad():
            yp = model(x_t).cpu().numpy()[0]

        if not found_no_sw and rng_vc < 0.5:
            plot_prediction_physical(va, vc, y_t, yp, stats_v, fs_v,
                                     title=f"{titulo_prefix} – SIN switching")
            found_no_sw = True

        if not found_sw and rng_vc > 3.0:
            plot_prediction_physical(va, vc, y_t, yp, stats_v, fs_v,
                                     title=f"{titulo_prefix} – CON switching")
            found_sw = True

    if not found_no_sw:
        print("  No se encontró ventana sin switching en test")
    if not found_sw:
        print("  No se encontró ventana con switching en test")
    print(f"\n Visualización {titulo_prefix} completada")


#LSTM
buscar_y_plotear(model, test_dataset, ckpt, device, "Test LSTM")


In [ ]:
# BLOQUE 12: Visualización de predicciones en escala física – LSTM
#
# MEJORAS respecto al original:
# 1. 4to subplot con error residual (real − pred) en mV para S1A y S1B
# 2. Real y pred con colores distintos (no solo línea vs punteado)
# 3. Ventana recortada alrededor del evento de switching (±20% del rango)
# 4. Criterio de selección de muestra: exige señal analógica con varianza
#    mínima para evitar ventanas con v_analog ≈ 0 todo el tiempo

def denorm(arr, key, stats):
    return arr * stats[key]["std"] + stats[key]["mean"]

def plot_prediction_physical(v_a_norm, v_c_norm, y_real_norm, y_pred_norm,
                              stats, fs, title=""):
    v_a   = denorm(v_a_norm,         "v_analog", stats)
    v_c   = denorm(v_c_norm,         "v_ctrl",   stats)
    s1a_r = denorm(y_real_norm[:,0], "s1a",      stats)
    s1b_r = denorm(y_real_norm[:,1], "s1b",      stats)
    s1a_p = denorm(y_pred_norm[:,0], "s1a",      stats)
    s1b_p = denorm(y_pred_norm[:,1], "s1b",      stats)

    T = len(v_a)
    t = np.arange(T) / fs * 1e6  # µs

    # Recorte temporal alrededor del switching
    # Si hay switching, centrar la ventana en el flanco. Si no, mostrar todo.
    ctrl_phys = v_c * stats["v_ctrl"]["std"] + stats["v_ctrl"]["mean"] \
                if False else denorm(v_c_norm, "v_ctrl", stats)
    is_on  = ctrl_phys > 2.5
    flancos = np.where(np.diff(is_on.astype(int)) != 0)[0]

    if len(flancos) > 0:
        # Centro en el primer flanco, margen de 30% de la ventana a cada lado
        centro = flancos[0]
        margen = max(int(0.30 * T), 20)
        i0 = max(0, centro - margen)
        i1 = min(T, centro + margen * 3)  # más cola post-switching para ver régimen
    else:
        i0, i1 = 0, T

    t_plot   = t[i0:i1]
    v_a_plot = v_a[i0:i1]
    v_c_plot = ctrl_phys[i0:i1]
    s1a_r_p  = s1a_r[i0:i1];  s1a_p_p = s1a_p[i0:i1]
    s1b_r_p  = s1b_r[i0:i1];  s1b_p_p = s1b_p[i0:i1]

    # Error residual en mV
    err_a = (s1a_r_p - s1a_p_p) * 1e3
    err_b = (s1b_r_p - s1b_p_p) * 1e3

    # Plot
    fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
    fig.suptitle(title, fontsize=12, fontweight="bold")

    # Subplot 1: señal analógica de entrada
    axes[0].plot(t_plot, v_a_plot, color="#2563EB", lw=1.0, label="v_analog")
    axes[0].set_ylabel("v_analog [V]")
    axes[0].set_ylim(-3.5, 3.5)
    axes[0].axhline(0, color="k", lw=0.4, ls="--", alpha=0.4)
    axes[0].legend(loc="upper right", fontsize=9)
    axes[0].grid(True, alpha=0.25)

    # Subplot 2: señal de control
    axes[1].plot(t_plot, v_c_plot, color="#D97706", lw=1.0, label="v_ctrl")
    axes[1].axhline(2.5, color="#DC2626", lw=0.8, ls="--", label="Vth = 2.5 V")
    axes[1].set_ylabel("v_ctrl [V]")
    axes[1].set_ylim(-0.3, 5.5)
    axes[1].legend(loc="upper right", fontsize=9)
    axes[1].grid(True, alpha=0.25)

    # Subplot 3: salidas real vs pred (colores distintos para real y pred)
    axes[2].plot(t_plot, s1a_r_p, color="#16A34A", lw=1.4,
                 label="S1A ground truth")
    axes[2].plot(t_plot, s1a_p_p, color="#86EFAC", lw=1.1, ls="--",
                 label="S1A ground truth")
    axes[2].plot(t_plot, s1b_r_p, color="#DC2626", lw=1.4,
                 label="S1B ground truth")
    axes[2].plot(t_plot, s1b_p_p, color="#FCA5A5", lw=1.1, ls="--",
                 label="S1B ground truth")
    axes[2].set_ylabel("Output [V]")
    axes[2].legend(loc="upper right", fontsize=9, ncol=2)
    axes[2].grid(True, alpha=0.25)

    # Subplot 4: error residual en mV
    axes[3].plot(t_plot, err_a, color="#16A34A", lw=1.0, label="S1A Error")
    axes[3].plot(t_plot, err_b, color="#DC2626", lw=1.0, label="S1B Error")
    axes[3].axhline(0, color="k", lw=0.5, ls="--", alpha=0.5)
    axes[3].set_ylabel("Error [mV]")
    axes[3].set_xlabel("Time [µs]")
    axes[3].legend(loc="upper right", fontsize=9)
    axes[3].grid(True, alpha=0.25)

    # Línea vertical en el flanco de switching
    if len(flancos) > 0:
        t_sw = t[flancos[0]]
        for ax in axes:
            ax.axvline(t_sw, color="#7C3AED", lw=0.8, ls=":", alpha=0.7)

    # RMSE en el título
    rmse_a = np.sqrt(np.mean((s1a_r_p - s1a_p_p)**2)) * 1e3
    rmse_b = np.sqrt(np.mean((s1b_r_p - s1b_p_p)**2)) * 1e3
    fig.suptitle(f"{title}   |   RMSE S1A = {rmse_a:.2f} mV  ·  S1B = {rmse_b:.2f} mV",
                 fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()


def buscar_y_plotear(model, dataset, ckpt, device, titulo_prefix):
    """Busca ventanas CON y SIN switching con señal analógica activa."""
    model.eval()
    fs_v    = dataset.metadata["fs"]
    stats_v = ckpt["stats"]
    found_no_sw = found_sw = False

    for win_entry in dataset.window_index:
        if found_no_sw and found_sw:
            break

        batch_idx, local_idx, start = win_entry
        end = start + dataset.window_size

        s       = dataset._get_batch(batch_idx)[local_idx]
        v_c_raw = s["x"]["v_ctrl"][start:end]
        v_a_raw = s["x"]["v_analog"][start:end]
        rng_vc  = float(v_c_raw.max() - v_c_raw.min())

        # NUEVO: exigir señal analógica con varianza mínima
        std_va = float(v_a_raw.std())
        if std_va < 0.05:   # descarta ventanas donde v_analog ≈ 0 todo el tiempo
            continue

        va = v_a_raw.copy()
        vc = v_c_raw.copy()
        y1 = s["y"]["s1a"][start:end].copy()
        y2 = s["y"]["s1b"][start:end].copy()

        va = (va - stats_v["v_analog"]["mean"]) / stats_v["v_analog"]["std"]
        vc = (vc - stats_v["v_ctrl"]["mean"])   / stats_v["v_ctrl"]["std"]
        y1 = (y1 - stats_v["s1a"]["mean"])      / stats_v["s1a"]["std"]
        y2 = (y2 - stats_v["s1b"]["mean"])      / stats_v["s1b"]["std"]

        x_t = torch.from_numpy(np.stack([va, vc], axis=1)).float().unsqueeze(0).to(device)
        y_t = np.stack([y1, y2], axis=1)

        with torch.no_grad():
            yp = model(x_t).cpu().numpy()[0]

        if not found_no_sw and rng_vc < 0.5:
            plot_prediction_physical(va, vc, y_t, yp, stats_v, fs_v,
                                     title=f"{titulo_prefix} – without switching")
            found_no_sw = True

        if not found_sw and rng_vc > 3.0:
            plot_prediction_physical(va, vc, y_t, yp, stats_v, fs_v,
                                     title=f"{titulo_prefix} – with switching")
            found_sw = True

    if not found_no_sw:
        print("  No se encontró ventana sin switching en test")
    if not found_sw:
        print("  No se encontró ventana con switching en test")
    print(f"\n Visualización {titulo_prefix} completada")


# LSTM
buscar_y_plotear(model, test_dataset, ckpt, device, "Test LSTM")

In [ ]:
# BLOQUE 12: Visualización de predicciones en escala física LSTM

def denorm(arr, key, stats):
    return arr * stats[key]["std"] + stats[key]["mean"]

def plot_prediction_physical(v_a_norm, v_c_norm, y_real_norm, y_pred_norm,
                              stats, fs, title=""):
    v_a   = denorm(v_a_norm,         "v_analog", stats)
    v_c   = denorm(v_c_norm,         "v_ctrl",   stats)
    s1a_r = denorm(y_real_norm[:,0], "s1a",      stats)
    s1b_r = denorm(y_real_norm[:,1], "s1b",      stats)
    s1a_p = denorm(y_pred_norm[:,0], "s1a",      stats)
    s1b_p = denorm(y_pred_norm[:,1], "s1b",      stats)

    T = len(v_a)
    t = np.arange(T) / fs * 1e6  # µs

    ctrl_phys = denorm(v_c_norm, "v_ctrl", stats)
    is_on  = ctrl_phys > 2.5
    flancos = np.where(np.diff(is_on.astype(int)) != 0)[0]

    if len(flancos) > 0:
        centro = flancos[0]
        margen = max(int(0.30 * T), 20)
        i0 = max(0, centro - margen)
        i1 = min(T, centro + margen * 3)
    else:
        i0, i1 = 0, T

    t_plot   = t[i0:i1]
    v_a_plot = v_a[i0:i1]
    v_c_plot = ctrl_phys[i0:i1]
    s1a_r_p  = s1a_r[i0:i1];  s1a_p_p = s1a_p[i0:i1]
    s1b_r_p  = s1b_r[i0:i1];  s1b_p_p = s1b_p[i0:i1]

    err_a = (s1a_r_p - s1a_p_p) * 1e3
    err_b = (s1b_r_p - s1b_p_p) * 1e3

    fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)

    axes[0].plot(t_plot, v_a_plot, color="#2563EB", lw=1.0, label="Analog input")
    axes[0].set_ylabel("v_analog [V]")
    axes[0].set_ylim(-3.5, 3.5)
    axes[0].axhline(0, color="k", lw=0.4, ls="--", alpha=0.4)
    axes[0].legend(loc="upper right", fontsize=9)
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(t_plot, v_c_plot, color="#D97706", lw=1.0, label="Control signal")
    axes[1].axhline(2.5, color="#DC2626", lw=0.8, ls="--", label="Vth = 2.5 V")
    axes[1].set_ylabel("v_ctrl [V]")
    axes[1].set_ylim(-0.3, 5.5)
    axes[1].legend(loc="upper right", fontsize=9)
    axes[1].grid(True, alpha=0.25)

    axes[2].plot(t_plot, s1a_r_p, color="#16A34A", lw=1.4, label="S1A ground truth")
    axes[2].plot(t_plot, s1a_p_p, color="#86EFAC", lw=1.1, ls="--", label="S1A predicted")
    axes[2].plot(t_plot, s1b_r_p, color="#DC2626", lw=1.4, label="S1B ground truth")
    axes[2].plot(t_plot, s1b_p_p, color="#FCA5A5", lw=1.1, ls="--", label="S1B predicted")
    axes[2].set_ylabel("Output [V]")
    axes[2].legend(loc="upper right", fontsize=9, ncol=2)
    axes[2].grid(True, alpha=0.25)

    axes[3].plot(t_plot, err_a, color="#16A34A", lw=1.0, label="S1A residual error")
    axes[3].plot(t_plot, err_b, color="#DC2626", lw=1.0, label="S1B residual error")
    axes[3].axhline(0, color="k", lw=0.5, ls="--", alpha=0.5)
    axes[3].set_ylabel("Error [mV]")
    axes[3].set_xlabel("Time [µs]")
    axes[3].legend(loc="upper right", fontsize=9)
    axes[3].grid(True, alpha=0.25)

    if len(flancos) > 0:
        t_sw = t[flancos[0]]
        for ax in axes:
            ax.axvline(t_sw, color="#7C3AED", lw=0.8, ls=":", alpha=0.7,
                       label="Switching event" if ax is axes[1] else "")

    rmse_a = np.sqrt(np.mean((s1a_r_p - s1a_p_p)**2)) * 1e3
    rmse_b = np.sqrt(np.mean((s1b_r_p - s1b_p_p)**2)) * 1e3
    fig.suptitle(f"{title}   |   RMSE S1A = {rmse_a:.2f} mV  ·  S1B = {rmse_b:.2f} mV",
                 fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()


def buscar_y_plotear(model, dataset, ckpt, device, titulo_prefix):
    """Finds windows WITH and WITHOUT switching and plots them."""
    model.eval()
    fs_v    = dataset.metadata["fs"]
    stats_v = ckpt["stats"]
    found_no_sw = found_sw = False

    for win_entry in dataset.window_index:
        if found_no_sw and found_sw:
            break

        batch_idx, local_idx, start = win_entry
        end = start + dataset.windo


In [ ]:
# @title
# BLOQUE 13: Visualización de muestras + exportación a LTspice
#
# Genera para cada muestra seleccionada:
#   v_analog.pwl   → señal de entrada analógica en formato PWL
#   v_ctrl.pwl     → señal de control digital en formato PWL
#   adg436_sim.asc → netlist LTspice con todos los parámetros del modelo

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

# ── Parámetros físicos del modelo (deben coincidir con ADG436Analytical) ─────
MP = {
    "Ron":       4.0,
    "Rload":  1000.0,
    "Vth":       2.5,
    "Vhyst":     0.05,
    "ton":     160e-9,
    "toff":    100e-9,
    "tau_on":  160e-9 / 2.197,
    "tau_off": 100e-9 / 2.197,
    "Q_ci":     10e-12,
    "C_load":  200e-12,
    "V_glitch": 10e-12 / 200e-12,
    "tau_ci":    5e-9,
    "noise_std": np.sqrt(4 * 1.38e-23 * 300 * 4.0 * 5e6),
}

META_PATH = "/content/drive/MyDrive/universal_filter_ngspice/datasets/adg436_20k_metadata_v2.pt"
metadata  = torch.load(META_PATH, weights_only=False)
fs        = metadata["fs"]
OUT_BASE  = "adg436_ltspice"

# ─────────────────────────────────────────────────────────────────────────────
def signal_to_pwl(t_arr, v_arr, filepath):
    os.makedirs(os.path.dirname(os.path.abspath(filepath)), exist_ok=True)
    with open(filepath, "w") as fh:
        for t, v in zip(t_arr, v_arr):
            fh.write(f"{t:.10e} {v:.6e}\n")

# ─────────────────────────────────────────────────────────────────────────────
def write_ltspice_asc(out_dir, sid, n_pts, type_a, type_d, mp, fs):
    Ron     = mp["Ron"]
    Rload   = mp["Rload"]
    Vth     = mp["Vth"]
    Vhyst   = mp["Vhyst"]
    tau_on  = mp["tau_on"]
    tau_off = mp["tau_off"]
    V_gl    = mp["V_glitch"]
    tau_ci  = mp["tau_ci"]
    t_step  = 1.0 / fs
    t_stop  = n_pts / fs
    tau_s   = 0.1
    Sv      = 4 * 1.38e-23 * 300 * Ron
    nd      = np.sqrt(Sv)
    att_db  = 20 * np.log10(Rload / (Ron + Rload))
    vn_rms  = mp["noise_std"]

    L = []
    L.append("Version 4")
    L.append("SHEET 1 2200 1600")
    L.append("WIRE 160 240 160 320");  L.append("WIRE 160 240 400 240")
    L.append("WIRE 160 480 160 560");  L.append("WIRE 400 240 680 240")
    L.append("WIRE 680 240 960 240");  L.append("WIRE 960 240 1200 240")
    L.append("WIRE 1200 240 1200 320"); L.append("WIRE 1200 480 1200 560")
    L.append("WIRE 160 640 160 720");  L.append("WIRE 160 640 400 640")
    L.append("WIRE 160 880 160 960")
    L.append("SYMBOL voltage 160 400 R0")
    L.append("SYMATTR InstName V_ANALOG")
    L.append("SYMATTR Value PWL file=v_analog.pwl")
    L.append("SYMBOL voltage 160 800 R0")
    L.append("SYMATTR InstName V_CTRL")
    L.append("SYMATTR Value PWL file=v_ctrl.pwl")
    L.append("FLAG 400 640 ctrl")
    L.append("IOPIN 400 640 BiDir")
    L.append("SYMBOL voltage 400 224 R90")
    L.append("WINDOW 0 -32 56 VBottom 2")
    L.append("WINDOW 3 32 56 VTop 2")
    L.append("SYMATTR InstName V_NOISE")
    L.append(f"SYMATTR Value NOISE(0 {nd:.6e} 0 {t_step:.6e})")
    L.append("SYMBOL res 820 224 R90")
    L.append("WINDOW 0 0 56 VBottom 2")
    L.append("WINDOW 3 32 56 VTop 2")
    L.append("SYMATTR InstName R_SW_A")
    L.append("SYMATTR Value {Ron_sw}")
    sw_on  = f"1 / (1 + exp(-(V(ctrl) - {Vth}) / {tau_s}))"
    sw_off = f"1 / (1 + exp( (V(ctrl) - {Vth}) / {tau_s}))"
    L.append("SYMBOL bv 540 390 R0")
    L.append("SYMATTR InstName B_RON")
    L.append(f"SYMATTR Value V = {Ron} * ({sw_on}) + 1e6 * ({sw_off})")
    L.append("SYMBOL voltage 1200 400 R0")
    L.append("SYMATTR InstName V_CI_A")
    L.append(f"SYMATTR Value PULSE(0 {V_gl:.5f} 0 {tau_ci:.4e} {5*tau_ci:.4e} {tau_ci:.4e} 1e9)")
    L.append("SYMATTR SpiceLine Rser=0.01")
    L.append("SYMBOL res 1060 224 R90")
    L.append("WINDOW 0 0 56 VBottom 2")
    L.append("WINDOW 3 32 56 VTop 2")
    L.append("SYMATTR InstName R_LOAD")
    L.append(f"SYMATTR Value {Rload}")
    L.append("FLAG 160 560 0"); L.append("FLAG 160 960 0"); L.append("FLAG 1200 560 0")
    L.append("FLAG 960 240 S1A"); L.append("IOPIN 960 240 Out")
    L.append(f".param Ron={Ron}")
    L.append(f".param Rload={Rload}")
    L.append(f".param Vth={Vth}")
    L.append(f".param tau_on={tau_on:.6e}")
    L.append(f".param tau_off={tau_off:.6e}")
    L.append(f".param V_glitch={V_gl:.6e}")
    L.append(f".param tau_ci={tau_ci:.6e}")
    L.append(f".param noise_density={nd:.6e}")
    L.append(f".tran {t_step:.6e} {t_stop:.6e} 0 {t_step/10:.6e}")
    L.append(".backanno")
    L.append(".end")
    L.append("TEXT 1420 80  Left 2 ;=== ADG436 – Parámetros de simulación ===")
    L.append(f"TEXT 1420 120 Left 2 ;Ron           = {Ron} ohm")
    L.append(f"TEXT 1420 150 Left 2 ;Rload         = {Rload} ohm")
    L.append(f"TEXT 1420 180 Left 2 ;Atenuacion ON = {att_db:.4f} dB")
    L.append(f"TEXT 1420 210 Left 2 ;Vth           = {Vth} V  (+-{Vhyst*1e3:.0f} mV histeresis)")
    L.append(f"TEXT 1420 240 Left 2 ;ton           = {mp['ton']*1e9:.0f} ns  tau_RC = {tau_on*1e9:.2f} ns")
    L.append(f"TEXT 1420 270 Left 2 ;toff          = {mp['toff']*1e9:.0f} ns  tau_RC = {tau_off*1e9:.2f} ns")
    L.append(f"TEXT 1420 300 Left 2 ;Q_ci          = {mp['Q_ci']*1e12:.0f} pC  Vglitch = {V_gl*1e3:.1f} mV  tau = {tau_ci*1e9:.0f} ns")
    L.append(f"TEXT 1420 330 Left 2 ;Vn_rms        = {vn_rms*1e6:.4f} uV  (Johnson-Nyquist)")
    L.append(f"TEXT 1420 360 Left 2 ;Sv_density    = {nd*1e9:.4f} nV/rtHz")
    L.append(f"TEXT 1420 390 Left 2 ;Fs            = {fs/1e6:.0f} MHz  |  N_pts = {n_pts}")
    L.append(f"TEXT 1420 420 Left 2 ;Sample ID     = {sid}")
    L.append(f"TEXT 1420 450 Left 2 ;Tipo entrada  = {type_a}  /  {type_d}")

    asc_path = os.path.join(out_dir, "adg436_sim.asc")
    with open(asc_path, "w") as fh:
        fh.write("\n".join(L))
    return asc_path

# ── Seleccionar samples CON y SIN switching usando lazy loading ───────────────
def find_samples_lazy(dataset, seed=55):
    rng = np.random.default_rng(seed)
    # Obtener global_idx únicos de los samples en este split
    global_ids = list(dataset.split_indices)
    rng.shuffle(global_ids)
    with_sw, without_sw = [], []
    for g_idx in global_ids:
        if len(with_sw) >= 1 and len(without_sw) >= 1:
            break
        s  = load_sample_lazy(dataset, g_idx)
        vc = s["x"]["v_ctrl"]
        rv = float(np.max(vc) - np.min(vc))
        if rv > 3.0 and len(with_sw) < 1:
            with_sw.append(int(g_idx))
        elif rv < 0.5 and len(without_sw) < 1:
            without_sw.append(int(g_idx))
    return with_sw + without_sw

sample_ids = find_samples_lazy(train_dataset, seed=55)
print(f"Samples seleccionados: {sample_ids}")

# ── Visualización + exportación ───────────────────────────────────────────────
exported = []
fig = plt.figure(figsize=(16, 10))
fig.suptitle("ADG436 – Muestras para LTspice (escala física)", fontsize=13, fontweight="bold")
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.48, wspace=0.32)

for col, s_idx in enumerate(sample_ids):
    s      = load_sample_lazy(train_dataset, s_idx)
    n      = s["metadata"]["n_samples"]
    t      = np.arange(n) / fs
    t_us   = t * 1e6
    v_a    = s["x"]["v_analog"].astype(np.float64)
    v_c    = s["x"]["v_ctrl"].astype(np.float64)
    s1a    = s["y"]["s1a"].astype(np.float64)
    s1b    = s["y"]["s1b"].astype(np.float64)
    type_a = s["x"]["v_analog_type"]
    type_d = s["x"]["v_ctrl_type"]
    rv     = float(v_c.max() - v_c.min())
    label  = "CON switching" if rv > 3.0 else "SIN switching"
    att_db = 20 * np.log10(MP["Rload"] / (MP["Ron"] + MP["Rload"]))

    ax0 = fig.add_subplot(gs[0, col])
    ax0.plot(t_us, v_a, lw=0.7, color="C0")
    ax0.set_title(f"Sample {s_idx} – {label}\nv_analog: {type_a}", fontsize=9)
    ax0.set_ylabel("V_analog [V]"); ax0.set_ylim(-3.2, 3.2)
    ax0.axhline(0, color="k", lw=0.3, ls="--"); ax0.grid(True, alpha=0.25)

    ax1 = fig.add_subplot(gs[1, col])
    ax1.plot(t_us, v_c, lw=0.7, color="C1")
    ax1.axhline(MP["Vth"], color="r", lw=0.8, ls="--", label=f"Vth={MP['Vth']} V")
    ax1.axhline(MP["Vth"] + MP["Vhyst"], color="r", lw=0.4, ls=":", alpha=0.5)
    ax1.axhline(MP["Vth"] - MP["Vhyst"], color="r", lw=0.4, ls=":", alpha=0.5, label="±Vhyst")
    ax1.set_title(f"v_ctrl: {type_d}", fontsize=9)
    ax1.set_ylabel("V_ctrl [V]"); ax1.set_ylim(-0.3, 5.5)
    ax1.legend(fontsize=7); ax1.grid(True, alpha=0.25)

    ax2 = fig.add_subplot(gs[2, col])
    ax2.plot(t_us, s1a, lw=0.8, color="C2", label="S1A")
    ax2.plot(t_us, s1b, lw=0.8, color="C3", label="S1B", alpha=0.85)
    ax2.set_ylabel("Salida [V]"); ax2.set_xlabel("Tiempo [µs]")
    ax2.set_ylim(-3.2, 3.2); ax2.legend(fontsize=7); ax2.grid(True, alpha=0.25)

    info = (f"Ron={MP['Ron']} Ω  Rload={MP['Rload']} Ω  Att={att_db:.3f} dB\n"
            f"ton={MP['ton']*1e9:.0f} ns / toff={MP['toff']*1e9:.0f} ns  "
            f"CI={MP['Q_ci']*1e12:.0f} pC→{MP['V_glitch']*1e3:.1f} mV\n"
            f"Vth={MP['Vth']} V ±{MP['Vhyst']*1e3:.0f} mV  "
            f"Vn={MP['noise_std']*1e6:.3f} µV rms")
    ax2.text(0.01, 0.02, info, transform=ax2.transAxes, fontsize=6.5,
             va="bottom", family="monospace",
             bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.88))

    out_dir = f"{OUT_BASE}_sample{s_idx}"
    os.makedirs(out_dir, exist_ok=True)
    signal_to_pwl(t, v_a, os.path.join(out_dir, "v_analog.pwl"))
    signal_to_pwl(t, v_c, os.path.join(out_dir, "v_ctrl.pwl"))
    asc_path = write_ltspice_asc(out_dir, s_idx, n, type_a, type_d, MP, fs)

    exported.append({"sample_id": s_idx, "label": label, "out_dir": out_dir,
                     "asc_path": asc_path, "n_pts": n, "t_us": n/fs*1e6,
                     "att_db": att_db, "type_analog": type_a, "type_ctrl": type_d})

plt.tight_layout()
plt.show()

# ── Resumen ───────────────────────────────────────────────────────────────────
print("\n" + "="*72)
print("ARCHIVOS EXPORTADOS PARA LTSPICE")
print("="*72)
for e in exported:
    print(f"\n📁 {e['out_dir']}/")
    print(f"   Sample ID    : {e['sample_id']}  ({e['label']})")
    print(f"   Señal entrada: {e['type_analog']}")
    print(f"   Señal control: {e['type_ctrl']}")
    print(f"   Duración     : {e['t_us']:.1f} µs  ({e['n_pts']:,} puntos @ {fs/1e6:.0f} MHz)")
    print(f"   Atenuación ON: {e['att_db']:.4f} dB")
    print(f"   ├── v_analog.pwl")
    print(f"   ├── v_ctrl.pwl")
    print(f"   └── adg436_sim.asc")

print("\n" + "="*72)
print("PARÁMETROS SPICE EXTRAÍDOS DEL MODELO ADG436")
print("="*72)
nd  = np.sqrt(4 * 1.38e-23 * 300 * MP["Ron"])
att = 20 * np.log10(MP["Rload"] / (MP["Ron"] + MP["Rload"]))
rows = [
    ("Ron",            f"{MP['Ron']} Ω",                    "resistencia ON, datasheet typ"),
    ("Rload",          f"{MP['Rload']} Ω",                  "carga de test"),
    ("Atenuación ON",  f"{att:.4f} dB",                     "20·log(Rload/(Ron+Rload))"),
    ("Vth",            f"{MP['Vth']} V",                    "umbral de conmutación"),
    ("Histéresis",     f"±{MP['Vhyst']*1e3:.0f} mV",        ""),
    ("ton",            f"{MP['ton']*1e9:.0f} ns",            "datasheet typ"),
    ("toff",           f"{MP['toff']*1e9:.0f} ns",           "datasheet typ"),
    ("tau_RC_on",      f"{MP['tau_on']*1e9:.2f} ns",         "ton/2.197"),
    ("tau_RC_off",     f"{MP['tau_off']*1e9:.2f} ns",        "toff/2.197"),
    ("Q_ci",           f"{MP['Q_ci']*1e12:.0f} pC",          "charge injection typ"),
    ("C_load",         f"{MP['C_load']*1e12:.0f} pF",        "capacidad de carga asumida"),
    ("V_glitch_pk",    f"{MP['V_glitch']*1e3:.1f} mV",       "Q_ci / C_load"),
    ("tau_ci",         f"{MP['tau_ci']*1e9:.0f} ns",         "decaimiento del glitch CI"),
    ("Vn_rms (5 MHz)", f"{MP['noise_std']*1e6:.4f} µV",      "Johnson-Nyquist sqrt(4kTR·BW)"),
    ("Sv density",     f"{nd*1e9:.4f} nV/√Hz",               "densidad espectral"),
]
for name, val, note in rows:
    note_str = f"  ← {note}" if note else ""
    print(f"  {name:<22} {val:<18}{note_str}")

print("="*72)
print("\n Abrir adg436_sim.asc en LTspice XVII")
print("   Los archivos .pwl deben estar en la misma carpeta que el .asc")


TCN

In [ ]:
# BLOQUE 14: Arquitectura TCN + configuración de entrenamiento
#
# TCN (Temporal Convolutional Network):
#   - Convoluciones dilatadas causales → campo receptivo exponencial
#   - Residual connections para estabilidad del gradiente
#   - Mismos hiperparámetros de entrenamiento que LSTM (BATCH_SIZE, LR, etc.)
#
# Referencia: Bai et al. (2018) "An Empirical Evaluation of Generic Convolutional
#             and Recurrent Networks for Sequence Modeling"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time


_wn = torch.nn.utils.parametrizations.weight_norm

class CausalConv1d(nn.Module):
    """Convolución causal 1D con padding automático."""
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad  = (kernel_size - 1) * dilation
        # weight_norm va aquí, sobre el Conv1d directamente (tiene atributo "weight")
        self.conv = _wn(nn.Conv1d(in_ch, out_ch, kernel_size,
                                  dilation=dilation, padding=self.pad))

    def forward(self, x):
        out = self.conv(x)
        return out[:, :, :-self.pad] if self.pad else out


class TCNBlock(nn.Module):
    """Bloque residual TCN: 2× (CausalConv → ReLU → Dropout)."""
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        self.conv1      = CausalConv1d(in_ch,  out_ch, kernel_size, dilation)
        self.conv2      = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.relu       = nn.ReLU()
        self.dropout    = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def _init_weights(self):
        self.conv1.weight_g.data.fill_(1)
        self.conv2.weight_g.data.fill_(1)

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.dropout(self.relu(self.conv1(x)))
        out = self.dropout(self.relu(self.conv2(out)))
        return self.relu(out + residual)


class ADG436TCN(nn.Module):
    """
    TCN para emulación del ADG436.
    Entrada : [v_analog, v_ctrl]  → 2 features
    Salida  : [s1a, s1b]          → 2 outputs

    Parámetros equivalentes a LSTM (hidden_size=128, num_layers=2):
      num_channels = [128, 128]  → mismo ancho
      kernel_size  = 3           → campo receptivo: (3-1)*(1+2)*2 = 12 muestras
      dropout      = 0.2
    """
    def __init__(self, input_size=2, num_channels=None,
                 kernel_size=3, dropout=0.2, output_size=2):
        super().__init__()
        if num_channels is None:
            num_channels = [128, 128]

        layers = []
        in_ch = input_size
        for i, out_ch in enumerate(num_channels):
            dilation = 2 ** i          # dilaciones: 1, 2, 4, …
            layers.append(TCNBlock(in_ch, out_ch, kernel_size, dilation, dropout))
            in_ch = out_ch

        self.network = nn.Sequential(*layers)
        self.fc      = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        # x: (batch, seq, features) → (batch, features, seq)
        out = self.network(x.permute(0, 2, 1))
        out = out.permute(0, 2, 1)         # → (batch, seq, channels)
        return self.fc(out)


# Hiperparámetros (idénticos a LSTM)
BATCH_SIZE    = 32
NUM_EPOCHS    = 50
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(device.type == "cuda"),
                          drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

model_tcn  = ADG436TCN().to(device)
num_params_tcn = sum(p.numel() for p in model_tcn.parameters() if p.requires_grad)
print(f"\nParámetros TCN: {num_params_tcn:,}")

criterion_tcn = nn.MSELoss()
optimizer_tcn = optim.Adam(model_tcn.parameters(), lr=LEARNING_RATE)
scheduler_tcn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_tcn, mode="min",
                                                      factor=0.5, patience=5)


In [ ]:
# BLOQUE 15: Loop de entrenamiento TCN

metadata   = torch.load("/content/drive/MyDrive/universal_filter_ngspice/datasets/adg436_20k_metadata_v2.pt",
                         weights_only=False)
stats_meta = metadata["stats"]

train_losses_tcn, val_losses_tcn = [], []
best_val_loss_tcn = float("inf")
best_epoch_tcn    = 0
start_time_tcn    = time.time()

print("=" * 70)
print("ENTRENAMIENTO ADG436 TCN")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    tl = train_epoch(model_tcn, train_loader, criterion_tcn, optimizer_tcn, device)
    vl = validate_epoch(model_tcn, val_loader, criterion_tcn, device)
    train_losses_tcn.append(tl)
    val_losses_tcn.append(vl)
    scheduler_tcn.step(vl)
    lr = optimizer_tcn.param_groups[0]["lr"]
    print(f"Epoch [{epoch:3d}/{NUM_EPOCHS}] Train: {tl:.8f} | Val: {vl:.8f} | LR: {lr:.6f}")

    if vl < best_val_loss_tcn:
        best_val_loss_tcn = vl
        best_epoch_tcn    = epoch
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     model_tcn.state_dict(),
            "optimizer_state_dict": optimizer_tcn.state_dict(),
            "train_loss":           tl,
            "val_loss":             vl,
            "stats":                stats_meta,
            "input_size":           2,
            "output_size":          2,
            "num_params":           num_params_tcn,
        }, "adg436_tcn_best.pt")
        print(f"    Mejor modelo guardado (Val: {vl:.8f})")

elapsed_tcn = time.time() - start_time_tcn
print(f"\n Entrenamiento completado en {elapsed_tcn/60:.2f} min")
print(f"   Mejor época   : {best_epoch_tcn}/{NUM_EPOCHS}")
print(f"   Mejor Val Loss: {best_val_loss_tcn:.8f}")

# Curva de aprendizaje
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(train_losses_tcn, label="Train Loss")
ax.semilogy(val_losses_tcn,   label="Val Loss")
ax.axvline(best_epoch_tcn - 1, color="r", ls="--", label=f"Best epoch {best_epoch_tcn}")
ax.set_xlabel("Época")
ax.set_ylabel("MSE (log)")
ax.set_title("Curva de aprendizaje ADG436 TCN")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# BLOQUE 16: Evaluación en test set TCN

ckpt_tcn = torch.load("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_tcn_best.pt", weights_only=False, map_location=device)
model_tcn.load_state_dict(ckpt_tcn["model_state_dict"])
model_tcn.eval()

stats_tcn = ckpt_tcn["stats"]
print(f"Checkpoint época  : {ckpt_tcn['epoch']}")
print(f"Val loss (norm)   : {ckpt_tcn['val_loss']:.8f}")

mse_sum  = 0.0
n_pts    = 0
mse_on_a = mse_off_a = mse_on_b = mse_off_b = 0.0
n_on     = n_off = 0
y_true_list_tcn, y_pred_list_tcn = [], []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Test TCN"):
        x, y  = x.to(device), y.to(device)
        yp    = model_tcn(x)

        mse_sum += nn.functional.mse_loss(yp, y, reduction="sum").item()
        n_pts   += y.numel()

        ctrl = x[:, :, 1]
        mon  = ctrl >  0.0
        moff = ctrl <= 0.0

        da = (yp[:,:,0] - y[:,:,0])**2
        db = (yp[:,:,1] - y[:,:,1])**2

        mse_on_a  += da[mon].sum().item()
        mse_off_a += da[moff].sum().item()
        mse_on_b  += db[mon].sum().item()
        mse_off_b += db[moff].sum().item()
        n_on  += mon.sum().item()
        n_off += moff.sum().item()

        y_true_list_tcn.append(y.cpu().numpy())
        y_pred_list_tcn.append(yp.cpu().numpy())

mse_norm_tcn  = mse_sum / n_pts
mse_on_a  /= max(n_on,  1)
mse_off_a /= max(n_off, 1)
mse_on_b  /= max(n_on,  1)
mse_off_b /= max(n_off, 1)

std_s1a    = stats_tcn["s1a"]["std"]
std_s1b    = stats_tcn["s1b"]["std"]
mse_phys_a = mse_norm_tcn * std_s1a**2
mse_phys_b = mse_norm_tcn * std_s1b**2

y_true_all_tcn = np.concatenate(y_true_list_tcn).reshape(-1, 2)
y_pred_all_tcn = np.concatenate(y_pred_list_tcn).reshape(-1, 2)

r2_a_tcn = r2(y_true_all_tcn[:,0], y_pred_all_tcn[:,0])
r2_b_tcn = r2(y_true_all_tcn[:,1], y_pred_all_tcn[:,1])

print("RESULTADOS EN TEST SET – TCN")
print(f"MSE global (norm)       : {mse_norm_tcn:.6e}")
print(f"MSE S1A (física, V²)    : {mse_phys_a:.6e}  → RMSE = {np.sqrt(mse_phys_a)*1e3:.3f} mV")
print(f"MSE S1B (física, V²)    : {mse_phys_b:.6e}  → RMSE = {np.sqrt(mse_phys_b)*1e3:.3f} mV")
print(f"R² S1A                  : {r2_a_tcn:.6f}")
print(f"R² S1B                  : {r2_b_tcn:.6f}")
print(f"\nMSE por estado (norm):")
print(f"  S1A ON  : {mse_on_a:.6e}")
print(f"  S1A OFF : {mse_off_a:.6e}")
print(f"  S1B ON  : {mse_on_b:.6e}")
print(f"  S1B OFF : {mse_off_b:.6e}")
if mse_off_a > 0:
    print(f"  Ratio ON/OFF S1A: {mse_on_a/mse_off_a:.2f}")
print("="*70)


In [ ]:
# BLOQUE 17: Visualización de predicciones en escala física TCN

buscar_y_plotear(model_tcn, test_dataset, ckpt_tcn, device, "Test TCN")


In [ ]:
buscar_y_plotear(model_tcn, test_dataset, ckpt, device, "Test TCN")

GRU

In [ ]:
# BLOQUE 18: Arquitectura RNN/GRU + configuración de entrenamiento
#
# Se usa GRU (Gated Recurrent Unit) en lugar de RNN vanilla:
#   - GRU es la variante recurrente más común ("RNN moderna")
#   - Menos parámetros que LSTM (sin cell state) pero similar capacidad
#   - Parámetros idénticos a LSTM: hidden_size=128, num_layers=2, dropout=0.2
#
# Por qué GRU y no RNN vanilla:
#   - RNN vanilla sufre de vanishing gradient en secuencias largas
#   - GRU soluciona este problema con gates (reset y update)
#   - Es el punto de comparación estándar entre "LSTM vs GRU vs TCN"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time


class ADG436GRU(nn.Module):
    """
    GRU (RNN moderna) para emulación del ADG436.
    Entrada : [v_analog, v_ctrl]  → 2 features
    Salida  : [s1a, s1b]          → 2 outputs

    Parámetros equivalentes a LSTM:
      hidden_size = 128
      num_layers  = 2
      dropout     = 0.2
    """
    def __init__(self, input_size=2, hidden_size=128,
                 num_layers=2, dropout=0.2, output_size=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out)


# Hiperparámetros (idénticos a LSTM y TCN)
BATCH_SIZE    = 32
NUM_EPOCHS    = 50
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(device.type == "cuda"),
                          drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

model_gru  = ADG436GRU().to(device)
num_params_gru = sum(p.numel() for p in model_gru.parameters() if p.requires_grad)
print(f"\nParámetros GRU: {num_params_gru:,}")

criterion_gru = nn.MSELoss()
optimizer_gru = optim.Adam(model_gru.parameters(), lr=LEARNING_RATE)
scheduler_gru = optim.lr_scheduler.ReduceLROnPlateau(optimizer_gru, mode="min",
                                                      factor=0.5, patience=5)


In [ ]:
# BLOQUE 19: Loop de entrenamiento GRU (RNN)

metadata   = torch.load("/content/drive/MyDrive/universal_filter_ngspice/datasets/adg436_20k_metadata_v2.pt",
                         weights_only=False)
stats_meta = metadata["stats"]

train_losses_gru, val_losses_gru = [], []
best_val_loss_gru = float("inf")
best_epoch_gru    = 0
start_time_gru    = time.time()

print("=" * 70)
print("ENTRENAMIENTO ADG436 GRU (RNN)")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    tl = train_epoch(model_gru, train_loader, criterion_gru, optimizer_gru, device)
    vl = validate_epoch(model_gru, val_loader, criterion_gru, device)
    train_losses_gru.append(tl)
    val_losses_gru.append(vl)
    scheduler_gru.step(vl)
    lr = optimizer_gru.param_groups[0]["lr"]
    print(f"Epoch [{epoch:3d}/{NUM_EPOCHS}] Train: {tl:.8f} | Val: {vl:.8f} | LR: {lr:.6f}")

    if vl < best_val_loss_gru:
        best_val_loss_gru = vl
        best_epoch_gru    = epoch
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     model_gru.state_dict(),
            "optimizer_state_dict": optimizer_gru.state_dict(),
            "train_loss":           tl,
            "val_loss":             vl,
            "stats":                stats_meta,
            "input_size":           2,
            "output_size":          2,
            "num_params":           num_params_gru,
        }, "adg436_gru_best.pt")
        print(f"   Mejor modelo guardado (Val: {vl:.8f})")

elapsed_gru = time.time() - start_time_gru
print(f"\n Entrenamiento completado en {elapsed_gru/60:.2f} min")
print(f"   Mejor época   : {best_epoch_gru}/{NUM_EPOCHS}")
print(f"   Mejor Val Loss: {best_val_loss_gru:.8f}")

# Curva de aprendizaje
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(train_losses_gru, label="Train Loss")
ax.semilogy(val_losses_gru,   label="Val Loss")
ax.axvline(best_epoch_gru - 1, color="r", ls="--", label=f"Best epoch {best_epoch_gru}")
ax.set_xlabel("Época")
ax.set_ylabel("MSE (log)")
ax.set_title("Curva de aprendizaje ADG436 GRU (RNN)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# BLOQUE 20: Evaluación en test set GRU (RNN)

ckpt_gru = torch.load("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_gru_best.pt", weights_only=False, map_location=device)
model_gru.load_state_dict(ckpt_gru["model_state_dict"])
model_gru.eval()

stats_gru = ckpt_gru["stats"]
print(f"Checkpoint época  : {ckpt_gru['epoch']}")
print(f"Val loss (norm)   : {ckpt_gru['val_loss']:.8f}")

mse_sum  = 0.0
n_pts    = 0
mse_on_a = mse_off_a = mse_on_b = mse_off_b = 0.0
n_on     = n_off = 0
y_true_list_gru, y_pred_list_gru = [], []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Test GRU"):
        x, y  = x.to(device), y.to(device)
        yp    = model_gru(x)

        mse_sum += nn.functional.mse_loss(yp, y, reduction="sum").item()
        n_pts   += y.numel()

        ctrl = x[:, :, 1]
        mon  = ctrl >  0.0
        moff = ctrl <= 0.0

        da = (yp[:,:,0] - y[:,:,0])**2
        db = (yp[:,:,1] - y[:,:,1])**2

        mse_on_a  += da[mon].sum().item()
        mse_off_a += da[moff].sum().item()
        mse_on_b  += db[mon].sum().item()
        mse_off_b += db[moff].sum().item()
        n_on  += mon.sum().item()
        n_off += moff.sum().item()

        y_true_list_gru.append(y.cpu().numpy())
        y_pred_list_gru.append(yp.cpu().numpy())

mse_norm_gru  = mse_sum / n_pts
mse_on_a  /= max(n_on,  1)
mse_off_a /= max(n_off, 1)
mse_on_b  /= max(n_on,  1)
mse_off_b /= max(n_off, 1)

std_s1a    = stats_gru["s1a"]["std"]
std_s1b    = stats_gru["s1b"]["std"]
mse_phys_a = mse_norm_gru * std_s1a**2
mse_phys_b = mse_norm_gru * std_s1b**2

y_true_all_gru = np.concatenate(y_true_list_gru).reshape(-1, 2)
y_pred_all_gru = np.concatenate(y_pred_list_gru).reshape(-1, 2)

r2_a_gru = r2(y_true_all_gru[:,0], y_pred_all_gru[:,0])
r2_b_gru = r2(y_true_all_gru[:,1], y_pred_all_gru[:,1])

print("RESULTADOS EN TEST SET – GRU (RNN)")
print(f"MSE global (norm)       : {mse_norm_gru:.6e}")
print(f"MSE S1A (física, V²)    : {mse_phys_a:.6e}  → RMSE = {np.sqrt(mse_phys_a)*1e3:.3f} mV")
print(f"MSE S1B (física, V²)    : {mse_phys_b:.6e}  → RMSE = {np.sqrt(mse_phys_b)*1e3:.3f} mV")
print(f"R² S1A                  : {r2_a_gru:.6f}")
print(f"R² S1B                  : {r2_b_gru:.6f}")
print(f"\nMSE por estado (norm):")
print(f"  S1A ON  : {mse_on_a:.6e}")
print(f"  S1A OFF : {mse_off_a:.6e}")
print(f"  S1B ON  : {mse_on_b:.6e}")
print(f"  S1B OFF : {mse_off_b:.6e}")
if mse_off_a > 0:
    print(f"  Ratio ON/OFF S1A: {mse_on_a/mse_off_a:.2f}")
print("="*70)


In [ ]:
# BLOQUE 21: Visualización de predicciones en escala física – GRU (RNN)

buscar_y_plotear(model_gru, test_dataset, ckpt_gru, device, "Test GRU")


In [ ]:
# BLOQUE 22: Comparación final – LSTM vs TCN vs GRU
#
# Resume los resultados de las tres arquitecturas en una tabla y
# superpone las curvas de aprendizaje para comparación directa.

print(f"{'COMPARACIÓN FINAL: LSTM vs TCN vs GRU':^80}")

# Cargar checkpoints
ckpt_lstm = torch.load("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_lstm_best.pt", weights_only=False, map_location=device)
ckpt_tcn  = torch.load("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_tcn_best.pt",  weights_only=False, map_location=device)
ckpt_gru  = torch.load("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_gru_best.pt",  weights_only=False, map_location=device)
print(ckpt_lstm.keys())
rows = [
    ("LSTM", ckpt_lstm, train_losses,     val_losses,     "C0"),
    ("TCN",  ckpt_tcn,  train_losses_tcn, val_losses_tcn, "C1"),
    ("GRU",  ckpt_gru,  train_losses_gru, val_losses_gru, "C2"),
]

print(f"\n{'Modelo':<8} {'Params':>10} {'Best Epoch':>12} {'Val Loss':>14} {'R² S1A':>10} {'R² S1B':>10}")
print("-" * 68)

# Re-calcular R² para cada modelo sobre el test set
def eval_model_r2(model, loader, device):
    model.eval()
    yt_list, yp_list = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            yp = model(x)
            yt_list.append(y.cpu().numpy())
            yp_list.append(yp.cpu().numpy())
    yt = np.concatenate(yt_list).reshape(-1, 2)
    yp = np.concatenate(yp_list).reshape(-1, 2)
    r2a = r2(yt[:,0], yp[:,0])
    r2b = r2(yt[:,1], yp[:,1])
    return r2a, r2b

model.load_state_dict(ckpt_lstm["model_state_dict"])
model_tcn.load_state_dict(ckpt_tcn["model_state_dict"])
model_gru.load_state_dict(ckpt_gru["model_state_dict"])

r2a_lstm, r2b_lstm = eval_model_r2(model,     test_loader, device)
r2a_tcn,  r2b_tcn  = eval_model_r2(model_tcn, test_loader, device)
r2a_gru,  r2b_gru  = eval_model_r2(model_gru, test_loader, device)

results = [
    ("LSTM", ckpt_lstm["num_params"], ckpt_lstm["epoch"], ckpt_lstm["val_loss"], r2a_lstm, r2b_lstm),
    ("TCN",  ckpt_tcn["num_params"],  ckpt_tcn["epoch"],  ckpt_tcn["val_loss"],  r2a_tcn,  r2b_tcn),
    ("GRU",  ckpt_gru["num_params"],  ckpt_gru["epoch"],  ckpt_gru["val_loss"],  r2a_gru,  r2b_gru),
]

for name, params, epoch, val_loss, r2a, r2b in results:
    print(f"{name:<8} {params:>10,} {epoch:>12} {val_loss:>14.8f} {r2a:>10.6f} {r2b:>10.6f}")

print("=" * 68)

# Curvas de aprendizaje superpuestas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparación de entrenamiento: LSTM vs TCN vs GRU", fontsize=12)

for ax, losses_type, title in [
    (axes[0], [train_losses, train_losses_tcn, train_losses_gru], "Train Loss"),
    (axes[1], [val_losses,   val_losses_tcn,   val_losses_gru],   "Val Loss"),
]:
    for (name, _, _, _, color), losses in zip(
            [("LSTM","","","","C0"),("TCN","","","","C1"),("GRU","","","","C2")],
            losses_type):
        ax.semilogy(losses, label=name, color=color)
    ax.set_xlabel("Época"); ax.set_ylabel("MSE (log)")
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n Comparación completada")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
from tqdm import tqdm

def denorm(arr, key, stats):
    return arr * stats[key]['std'] + stats[key]['mean']

stats = stats_meta

# ── Calcular MSE por estado para cada modelo ──────────────────────────────────
def compute_state_metrics(model, loader, stats, device):
    """
    Devuelve dict con MSE_on_a, MSE_off_a, MSE_on_b, MSE_off_b (en V²)
    y sus RMSE en mV.
    """
    model.eval()
    acc = {'on_a': 0.0, 'off_a': 0.0, 'on_b': 0.0, 'off_b': 0.0}
    cnt = {'on_a': 0,   'off_a': 0,   'on_b': 0,   'off_b': 0}

    with torch.no_grad():
        for x_batch, y_batch in tqdm(loader, desc=f'  Evaluando', leave=False):
            x_b = x_batch.to(device)
            y_pred_norm = model(x_b).cpu()

            # Desnormalizar a escala física
            std_a  = stats['s1a']['std'];  mean_a = stats['s1a']['mean']
            std_b  = stats['s1b']['std'];  mean_b = stats['s1b']['mean']
            std_vc = stats['v_ctrl']['std']; mean_vc = stats['v_ctrl']['mean']

            v_ctrl_phys = x_batch[:, :, 1] * std_vc + mean_vc   # (B, T)
            is_on       = (v_ctrl_phys > 2.5)                    # máscara booleana

            s1a_real = y_batch[:, :, 0] * std_a + mean_a
            s1b_real = y_batch[:, :, 1] * std_b + mean_b
            s1a_pred = y_pred_norm[:, :, 0] * std_a + mean_a
            s1b_pred = y_pred_norm[:, :, 1] * std_b + mean_b

            err_a = (s1a_real - s1a_pred) ** 2
            err_b = (s1b_real - s1b_pred) ** 2

            acc['on_a']  += err_a[is_on].sum().item()
            acc['off_a'] += err_a[~is_on].sum().item()
            acc['on_b']  += err_b[is_on].sum().item()
            acc['off_b'] += err_b[~is_on].sum().item()
            cnt['on_a']  += is_on.sum().item()
            cnt['off_a'] += (~is_on).sum().item()
            cnt['on_b']  += is_on.sum().item()
            cnt['off_b'] += (~is_on).sum().item()

    mse  = {k: acc[k] / max(cnt[k], 1) for k in acc}
    rmse = {k: np.sqrt(mse[k]) * 1000 for k in mse}   # en mV
    return rmse

print('Calculando métricas por estado...')
results = {}
for mname, model in [('LSTM', model), ('TCN', model_tcn), ('GRU', model_gru)]:
    print(f'  {mname}...')
    results[mname] = compute_state_metrics(model, test_loader, stats, device)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ADG436 — RMSE per Conmutation State: LSTM vs TCN vs GRU',
             fontsize=13, fontweight='bold')

model_names = ['LSTM', 'TCN', 'GRU']
x = np.arange(len(model_names))
width = 0.35
colors_on  = ['#1f77b4', '#2ca02c', '#9467bd']
colors_off = ['#aec7e8', '#98df8a', '#c5b0d5']

for ax, channel, key_on, key_off in [
    (ax1, 'S1A', 'on_a', 'off_a'),
    (ax2, 'S1B', 'on_b', 'off_b'),
]:
    rmse_on  = [results[m][key_on]  for m in model_names]
    rmse_off = [results[m][key_off] for m in model_names]
    ratio    = [on/off if off > 0 else 0 for on, off in zip(rmse_on, rmse_off)]

    bars_on  = ax.bar(x - width/2, rmse_on,  width, label='ON state',
                      color=colors_on,  alpha=0.85, edgecolor='white')
    bars_off = ax.bar(x + width/2, rmse_off, width, label='OFF state',
                      color=colors_off, alpha=0.85, edgecolor='white')

    # Valores sobre las barras
    for bar in bars_on:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    for bar in bars_off:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

    # Ratio ON/OFF como anotación
    ax2_twin = ax.twinx()
    ax2_twin.plot(x, ratio, 'ko--', linewidth=1.2, markersize=6, label='Ratio ON/OFF')
    ax2_twin.set_ylabel('Ratio ON/OFF RMSE', fontsize=9, color='black')
    ax2_twin.tick_params(axis='y', labelcolor='black')
    ax2_twin.set_ylim(0, max(ratio) * 1.5)
    for xi, r in zip(x, ratio):
        ax2_twin.annotate(f'{r:.1f}×', (xi, r), textcoords='offset points',
                          xytext=(0, 8), ha='center', fontsize=8, color='black')

    ax.set_title(f'Channel {channel}', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, fontsize=10)
    ax.set_ylabel('RMSE (mV)', fontsize=9)
    ax.set_xlabel('Architecture', fontsize=9)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, axis='y', alpha=0.3)
    ax.set_ylim(0, max(rmse_on) * 1.4)

plt.tight_layout()
plt.savefig('fig_on_off_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: fig_on_off_analysis.png')
print('\nResumen ratio ON/OFF:')
for mname in model_names:
    r_a = results[mname]['on_a'] / results[mname]['off_a']
    r_b = results[mname]['on_b'] / results[mname]['off_b']
    print(f'  {mname}: S1A = {r_a:.2f}×  |  S1B = {r_b:.2f}×')

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Asegúrate de que las clases de tus modelos (ADG436LSTM, etc.) estén definidas
# Si tienes TCN o GRU, sus clases deben estar cargadas en el entorno.

def load_trained_model(ckpt_path, model_class, device):
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    # Instanciamos con los parámetros por defecto de tus bloques de código
    model = model_class().to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint["stats"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ejemplo para cargar el LSTM (ajusta la ruta a tu Drive)
lstm_model, stats_lstm = load_trained_model("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_lstm_best.pt", ADG436LSTM, device)
tcn_model, stats_tcn = load_trained_model("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_tcn_best.pt", ADG436TCN, device)
gru_model, stats_gru = load_trained_model("/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_gru_best.pt", ADG436GRU, device)

In [ ]:
def plot_model_inference(model, dataset, stats, device, title="Model vs Ground Truth"):
    # Elegir una ventana aleatoria del set de test
    idx = np.random.randint(len(dataset))
    x_batch, y_true = dataset[idx] # x_batch: [seq_len, 2], y_true: [seq_len, 2]

    # Inferencia
    with torch.no_grad():
        x_input = x_batch.unsqueeze(0).to(device) # Batch size 1
        y_pred = model(x_input).squeeze(0).cpu().numpy()

    y_true = y_true.numpy()

    # Desnormalizar para ver voltajes reales (V)
    s1a_pred = y_pred[:, 0] * stats["s1a"]["std"] + stats["s1a"]["mean"]
    s1a_true = y_true[:, 0] * stats["s1a"]["std"] + stats["s1a"]["mean"]

    s1b_pred = y_pred[:, 1] * stats["s1b"]["std"] + stats["s1b"]["mean"]
    s1b_true = y_true[:, 1] * stats["s1b"]["std"] + stats["s1b"]["mean"]

    # Crear Gráficos
    fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Canal S1A
    ax[0].plot(s1a_true, label="NGSpice Target", color='black', lw=1.5)
    ax[0].plot(s1a_pred, label="Neuronal Prediction", color='cyan', linestyle='--', lw=1.5)
    ax[0].set_title(f"{title} - S1A output")
    ax[0].set_ylabel("Volts [V]")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    # Canal S1B
    ax[1].plot(s1b_true, label="NGSpice Target", color='black', lw=1.5)
    ax[1].plot(s1b_pred, label="Neuronal Prediction", color='orange', linestyle='--', lw=1.5)
    ax[1].set_title(f"S1B output")
    ax[1].set_ylabel("Volts [V]")
    ax[1].set_xlabel("Samples")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_model_inference(lstm_model, test_dataset, stats, device, "LSTM")

In [ ]:
plot_model_inference(tcn_model, test_dataset, stats, device, "TCN")

In [ ]:
plot_model_inference(gru_model, test_dataset, stats, device, "RNN")

In [ ]:
def plot_error_distribution(model, dataset, stats, device, title='a'):
    idx = np.random.randint(len(dataset))
    x_batch, y_true = dataset[idx]

    with torch.no_grad():
        x_input = x_batch.unsqueeze(0).to(device)
        y_pred = model(x_input).squeeze(0).cpu().numpy()

    y_true = y_true.numpy()

    # Error absoluto en escala normalizada (para comparar precisión pura)
    error_a = np.abs(y_true[:, 0] - y_pred[:, 0])
    error_b = np.abs(y_true[:, 1] - y_pred[:, 1])

    plt.figure(figsize=(12, 4))
    plt.plot(error_a, label="S1A Error", color='red', alpha=0.7)
    plt.plot(error_b, label="S1B Error", color='blue', alpha=0.7)
    plt.fill_between(range(len(error_a)), error_a, color='red', alpha=0.1)
    plt.title(f"{title}"" - Prediction Absolute Error per Sample")
    plt.ylabel("Error (Normalized)")
    plt.xlabel("Sample per Window")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()


plot_error_distribution(lstm_model, test_dataset, stats, device, 'LSTM')

In [ ]:
plot_error_distribution(tcn_model, test_dataset, stats, device, 'TCN')

In [ ]:
plot_error_distribution(gru_model, test_dataset, stats, device, 'RNN')

In [ ]:
def compare_models_from_ckpts(ckpt_paths):
    names = []
    mse_vals = []

    for path in ckpt_paths:
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        name = path.split('/')[-1].replace('.pt', '')
        names.append(name)
        mse_vals.append(ckpt['val_loss']) # Usamos el último val_loss guardado

    plt.figure(figsize=(8, 5))
    bars = plt.bar(names, mse_vals, color=['blue', 'green', 'red'])
    plt.yscale('log') # Escala logarítmica para ver mejor las diferencias pequeñas
    plt.ylabel("Final Validation MSE (Log Scale)")
    plt.title("Comparación de Precisión: LSTM vs TCN vs RNN")

    # Añadir etiquetas de texto sobre las barras
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval, f'{yval:.2e}', va='bottom', ha='center')

    plt.show()

# Rutas de ejemplo (ajusta a tus nombres de archivo)
paths = [
   "/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_lstm_best.pt",
   "/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_tcn_best.pt",
   "/content/drive/MyDrive/universal_filter_ngspice/checkpoints/adg436_gru_best.pt"
]
compare_models_from_ckpts(paths)